In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Getting Started With Spark using Python**


### The Python API


Spark is written in Scala, which compiles to Java bytecode, but you can write python code to communicate to the java virtual machine through a library called py4j.

## Objectives


In this notebook, we will go over the basics of Apache Spark and PySpark. We will start with creating the SparkContext and SparkSession. We then create an RDD and apply some basic transformations and actions. Finally we demonstrate the basics dataframes and SparkSQL.

After this notebook you will be able to:

* Create the SparkContext and SparkSession
* Create an RDD and apply some basic transformations and actions to RDDs
* Demonstrate the use of the basics Dataframes and SparkSQL


----


## Setup


For this notebook, we are going to be using Python and Spark (PySpark). These libraries should be installed in your local environment.


In [1]:
# Installing required packages
# PySpark is the Spark API for Python.
!pip install pyspark
!pip install findspark

In [2]:
import findspark

# findspark is a Python library that allows you to easily locate and initialize a PySpark installation.
# It’s particularly useful when you want to run PySpark from outside of a managed environment
 # (like a cluster or Jupyter notebook) or when you're using PySpark in an environment where Spark
 # isn't installed globally (such as on a local machine with a specific Spark installation)

In [3]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [4]:

findspark.init()
# This initializes the findspark module, which:
  #Locates the installed Spark directory (either from your environment or by searching).
  #Sets up the environment variables required by PySpark, such as:
    #1. SPARK_HOME: The directory where Apache Spark is installed.
    #2. PYTHONPATH: Ensures that the PySpark library is included in your Python path.

## Exercise 1 -  Spark Context and Spark Session


In this exercise, we will create the Spark Context and initialize the Spark session needed for SparkSQL and DataFrames.
SparkContext is the entry point for Spark applications and contains functions to create RDDs such as `parallelize()`. SparkSession is needed for SparkSQL and DataFrame operations.


#### Task 1: Creating the spark session and context


In [5]:
# Creating a spark context class
sc = SparkContext()

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Python Spark DataFrames basic example") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

# "spark.some.config.option" is just an example placeholder and does not affect the actual
# Spark behavior unless there is a real configuration option specified.

# spark = SparkSession \.builder \.appName("My App") \.config("spark.executor.memory", "2g")
# \.config("spark.sql.shuffle.partitions", "50") \.getOrCreate()
# Here:
# "spark.executor.memory" allocates 2 GB of memory to each Spark executor.
# "spark.sql.shuffle.partitions" sets 50 partitions for SQL shuffle operations, impacting performance.

In [6]:
spark

#### Task 2: Initialize Spark session
To work with PySpark we just need to verify that the spark session instance has been created.


In [7]:
if 'spark' in locals() and isinstance(spark, SparkSession):
    print("SparkSession is active and ready to use.")
else:
    print("SparkSession is not active. Please create a SparkSession.")

SparkSession is active and ready to use.


## Exercise 2: RDDs
In this exercise we work with Resilient Distributed Datasets (RDDs). RDDs are Spark's primitive data abstraction and we use concepts from functional programming to create and manipulate RDDs.


#### Task 1: Create an RDD.
For demonstration purposes, we create an RDD here by calling `sc.parallelize()`  
We create an RDD which has integers from 1 to 30.


In [8]:
data = range(1,30)
# print first element of iterator
print(data[1])
len(data)


2


29

In [9]:
print(data)

range(1, 30)


In [10]:
xrangeRDD = sc.parallelize(data, 8)
# sc.parallelize(data, 4): Creates an RDD from the data collection with 4 partitions


xrangeRDD
# this will let us know that we created an RDD

PythonRDD[1] at RDD at PythonRDD.scala:56

In [11]:
xrangeRDD.count()

29

In [12]:
partitions = xrangeRDD.glom()
partitions.collect()

[[1, 2, 3],
 [4, 5, 6, 7],
 [8, 9, 10],
 [11, 12, 13, 14],
 [15, 16, 17, 18],
 [19, 20, 21],
 [22, 23, 24, 25],
 [26, 27, 28, 29]]

In [13]:
# Use glom() to view elements in each partition
partitions = xrangeRDD.glom().collect()

# Display the partitions
print("Partitions with their respective elements:")
for i, partition in enumerate(partitions):
    print(f"Partition {i}: {partition}")

Partitions with their respective elements:
Partition 0: [1, 2, 3]
Partition 1: [4, 5, 6, 7]
Partition 2: [8, 9, 10]
Partition 3: [11, 12, 13, 14]
Partition 4: [15, 16, 17, 18]
Partition 5: [19, 20, 21]
Partition 6: [22, 23, 24, 25]
Partition 7: [26, 27, 28, 29]


#### Task 2: Transformations


A transformation is an operation on an RDD that results in a new RDD. The transformed RDD is generated rapidly because the new RDD is lazily evaluated, which means that the calculation is not carried out when the new RDD is generated. The RDD will contain a series of transformations, or computation instructions, that will only be carried out when an action is called. In this transformation, we reduce each element in the RDD by 1. Note the use of the lambda function. We also then filter the RDD to only contain elements <10.


In [14]:
xrangeRDD.collect()

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29]

In [15]:
subRDD = xrangeRDD.map(lambda x: x*x)
subRDD.collect()

[1,
 4,
 9,
 16,
 25,
 36,
 49,
 64,
 81,
 100,
 121,
 144,
 169,
 196,
 225,
 256,
 289,
 324,
 361,
 400,
 441,
 484,
 529,
 576,
 625,
 676,
 729,
 784,
 841]

In [16]:
filteredRDD = subRDD.filter(lambda x : x<10)
filteredRDD.collect()

# Important: collect() should be used cautiously with large RDDs,
# as it brings all data to the driver, which could exhaust memory if the RDD is large.

[1, 4, 9]

#### Task 3: Actions


A transformation returns a result to the driver. We now apply the `collect()` action to get the output from the transformation.


In [17]:
#print(filteredRDD.collect())
filteredRDD.count()

3

#### Task 4: Caching Data


This simple example shows how to create an RDD and cache it. Notice the **speed improvement**!  If you wish to see the actual computation time, browse to the Spark UI...it's at host:4040.  You'll see that the second calculation took much less time!


In [18]:
testRDD = sc.parallelize(range(1,50000),4)
partitions = testRDD.glom()
partitions.collect()

[[1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  157,
  158,
  

In [19]:
import time

test = sc.parallelize(range(1,50000),4)
test.cache()
# The purpose of caching is to avoid re-computation of the RDD when it’s accessed multiple times.
# This means the RDD will be stored in memory the first time it’s evaluated, making future operations
# on it faster since Spark can retrieve it from memory instead of recalculating it.

t1 = time.time() # # Record the current time
# first count will trigger evaluation of count *and* cache
count1 = test.count()
dt1 = time.time() - t1 # Measure the time taken for the first count operation
print("dt1: ", dt1)


t2 = time.time()
# second count operates on cached data only
count2 = test.count()
dt2 = time.time() - t2
print("dt2: ", dt2)

#test.count()



dt1:  2.3597536087036133
dt2:  0.8118376731872559


**Summary**
**First count (dt1):** Includes both the RDD computation and caching time.

**Second count (dt2):** Much faster as it operates on the cached RDD in memory.

**Benefit:** Caching is useful when you need to perform multiple actions on the same RDD, as it reduces computation time by avoiding recalculations.

## Exercise 3: DataFrames and SparkSQL


In order to work with the extremely powerful SQL engine in Apache Spark, you will need a Spark Session. We have created that in the first Exercise, let us verify that spark session is still active.


In [20]:
spark

#### Task 1: Create Your First DataFrame!


You can create a structured data set (much like a database table) in Spark.  Once you have done that, you can then use powerful SQL tools to query and join your dataframes.


In [21]:
# df = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/BigData/Jupyter Notebook/Spark/people.csv", header = True).cache()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/drive/MyDrive/Colab Notebooks/BigData/Jupyter Notebook/Spark/people.csv. SQLSTATE: 42K03

In [22]:
df = spark.read.csv("people.csv", header = True).cache()

In [23]:
df

DataFrame[age: string, name: string]

In [24]:
# Print the dataframe as well as the data schema
df.show()


+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



In [25]:
df.printSchema()

root
 |-- age: string (nullable = true)
 |-- name: string (nullable = true)



In [26]:
# Get the number of rows
num_rows = df.count()

# Get the number of columns
num_columns = len(df.columns)

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_columns}")


Number of rows: 3
Number of columns: 2


In [27]:
# Register the DataFrame as a SQL temporary view
df.createTempView("people")

#### Task 2: Explore the data using DataFrame functions and SparkSQL

In this section, we explore the datasets using functions both from dataframes as well as corresponding SQL queries using sparksql. Note the different ways to achieve the same task!


In [28]:
# Select and show basic data columns

df.select("name").show()


+-------+
|   name|
+-------+
|Michael|
|   Andy|
| Justin|
+-------+



In [29]:
spark.sql("SELECT name FROM people").show()

+-------+
|   name|
+-------+
|Michael|
|   Andy|
| Justin|
+-------+



In [30]:
# Perform basic filtering

df.filter(df["age"] > 21).show()


+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



In [31]:
spark.sql("SELECT age, name FROM people WHERE age > 21").show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



In [32]:
# Perfom basic aggregation of data

df.groupBy("age").count().show()


+----+-----+
| age|count|
+----+-----+
|  30|    1|
|NULL|    1|
|  19|    1|
+----+-----+



In [33]:
spark.sql("SELECT age, COUNT(age) as count FROM people GROUP BY age").show()

+----+-----+
| age|count|
+----+-----+
|  30|    1|
|NULL|    0|
|  19|    1|
+----+-----+



In [34]:
spark.stop() # stop the current spark session

In [35]:
df.groupBy("age").count().show()

PySparkRuntimeError: [SESSION_OR_CONTEXT_NOT_EXISTS] SparkContext or SparkSession should be created first.

----


### Question 1 - RDDs


Create an RDD with integers from 1-50. Apply a transformation to multiply every number by 2, resulting in an RDD that contains the first 50 even numbers.


### Question 2 - DataFrames and SparkSQL


Similar to the `people.csv` file, now read the `people2.csv` file into the notebook, load it into a dataframe and apply SQL operations to determine the average age in our people2 file.


### Question 3 - SparkSession


Close the SparkSession we created for this notebook
